# 1. Setup - Installing the required libraries

In [1]:
!pip install transformers trl datasets huggingface_hub bitsandbytes peft> /dev/null 2>&1 #given for not getting the 100 lines output

In [2]:
!pip install wandb
#!wandb login

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 92.6 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: gitpython
    Found existing installation: GitPython 0.3.6
    Uninstalling GitPython-0.3.6:
      Successfully uninstalled GitPython-0.3.6━━━━━━━━━━━━━━━━━━━━ 1/3 [gitpython]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [wandb]32m2/3 [wandb]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pypi-publisher 0.0.4 requires gitpython==0.3.6, but you have gitpython 3.1.45 which is incompatible.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
Aborted!


# 2. Prepare the dataset

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Nadiveedishravanreddy/KernelBook-messages")

README.md:   0%|          | 0.00/741 [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/63.3M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/67.0M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/68.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18162 [00:00<?, ? examples/s]

In [4]:
dataset=ds['train'][0]

In [5]:
dataset

{'entry_point': 'SumAggregator',
 'original_triton_code': '# AOT ID: [\'0_inference\']\nfrom ctypes import c_void_p, c_long, c_int\nimport torch\nimport math\nimport random\nimport os\nimport tempfile\nfrom math import inf, nan\nfrom torch._inductor.hooks import run_intermediate_hooks\nfrom torch._inductor.utils import maybe_profile\nfrom torch._inductor.codegen.memory_planning import _align as align\nfrom torch import device, empty_strided\nfrom torch._inductor.async_compile import AsyncCompile\nfrom torch._inductor.select_algorithm import extern_kernels\nfrom torch._inductor.codegen.multi_kernel import MultiKernelCall\nimport triton\nimport triton.language as tl\nfrom torch._inductor.runtime.triton_heuristics import grid, split_scan_grid, grid_combo_kernels, start_graph, end_graph\nfrom torch._C import _cuda_getCurrentRawStream as get_raw_stream\n\naten = torch.ops.aten\ninductor_ops = torch.ops.inductor\n_quantized = torch.ops._quantized\nassert_size_stride = torch._C._dynamo.guards

In [17]:
row = dataset[0]
row

{'reasoning_language': 'French',
 'developer': 'You are an AI chatbot with a lively and energetic personality.',
 'user': 'Can you show me the latest trends on Twitter right now?',
 'analysis': "D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.\n\nJe devrais préciser que les tendances Twitter évoluent rapidement et sont spécifiques à chaque région. Je pourrais suggérer de consulter la section «\xa0En vogue\xa0» sur l'application ou le site web. Aussi, l'utilisation de hashtags et le suivi d'utilisateurs pertinents pourraient être utiles. Il est important de souligner que les tendances varient selon la région et l'heure de la journée. Je devrais garder un ton amical et 

# 3. Prepare the model

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
ds

DatasetDict({
    train: Dataset({
        features: ['entry_point', 'original_triton_code', 'python_code', 'triton_code', 'repo_name', 'module_name', 'synthetic', 'uuid', 'licenses', 'stars', 'sha', 'repo_link', 'messages'],
        num_rows: 18162
    })
})

In [ ]:
messages = ds['train']["triton_code"][0]
conversation = tokenizer.apply_chat_template(messages, tokenize=False)
print(conversation)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-24

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|>


In [8]:
from huggingface_hub import snapshot_download

# Download and save to given path
snapshot_download(
    repo_id="openai/gpt-oss-20b",
    local_dir="/jupyter-tutorial/hf_models/openai/gpt-oss-20b",
    local_dir_use_symlinks=False  # ensures real files, not symlinks
)


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

USAGE_POLICY:   0%|          | 0.00/200 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

dtypes.json: 0.00B [00:00, ?B/s]

original/model.safetensors:   0%|          | 0.00/13.8G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

'/jupyter-tutorial/hf_models/openai/gpt-oss-20b'

In [10]:
!!git clone --recurse https://github.com/ROCm/bitsandbytes && cd bitsandbytes && git checkout rocm_enabled_multi_backend && pip install -r requirements-dev.txt && cmake -DCOMPUTE_BACKEND=hip -S . && make -j  && pip install .

["Cloning into 'bitsandbytes'...",
 "Switched to a new branch 'rocm_enabled_multi_backend'",
 "branch 'rocm_enabled_multi_backend' set up to track 'origin/rocm_enabled_multi_backend'.",
 'Requirement already satisfied: setuptools>=63 in /opt/conda/envs/py_3.12/lib/python3.12/site-packages (from -r requirements-dev.txt (line 2)) (80.1.0)',
 'Collecting pytest~=8.3.1 (from -r requirements-dev.txt (line 3))',
 '  Downloading pytest-8.3.5-py3-none-any.whl.metadata (7.6 kB)',
 'Collecting einops~=0.8.0 (from -r requirements-dev.txt (line 4))',
 '  Downloading einops-0.8.1-py3-none-any.whl.metadata (13 kB)',
 'Collecting wheel~=0.43.0 (from -r requirements-dev.txt (line 5))',
 '  Downloading wheel-0.43.0-py3-none-any.whl.metadata (2.2 kB)',
 'Collecting lion-pytorch~=0.2.2 (from -r requirements-dev.txt (line 6))',
 '  Downloading lion_pytorch-0.2.3-py3-none-any.whl.metadata (616 bytes)',
 'Requirement already satisfied: scipy~=1.14.0 in /opt/conda/envs/py_3.12/lib/python3.12/site-packages (f

In [9]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b",**model_kwargs)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

bitsandbytes library load error: Configured ROCm binary not found at /opt/conda/envs/py_3.12/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_rocm64.so
 If you are using Intel CPU/XPU, please install intel_extension_for_pytorch to enable required ops
Traceback (most recent call last):
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 318, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/py_3.12/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 282, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured ROCm binary not found at /opt/conda/envs/py_3.12/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_rocm64.so


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [ ]:
def combine_text(example):
    prefix_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
    )

    if example["module_name"] and example["module_name"] != "Not applicable":
        prompt = (
            f"<start_of_turn>user {prefix_text}Here are the inputs: {example['module_name']} <end_of_turn>\n"
        )
    else:
        prompt = (
            f"<start_of_turn>user {prefix_text}No module name provided <end_of_turn>\n"
        )

    # The output the model should generate is the Triton code
    completion = f"<start_of_turn>model {example['triton_code']} <end_of_turn>"

    return {
        "prompt": prompt,
        "completion": completion
    }
ds = ds.map(combine_text)

# 4. Fine-tuning

In [11]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 15,040,512 || all params: 20,929,797,696 || trainable%: 0.0719


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/peft/tuners/lora/layer.py:159: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(


In [28]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=2048,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir="checkpoints/demo/sft",
    report_to="wandb",
    run_name="gpt-oss-20b-sft",
    push_to_hub=True,
    hub_model_id="Nadiveedishravanreddy/gpt-oss-20b-triton-kernel",
    dataset_text_field='messages',


)

In [29]:
from huggingface_hub import login

login(token="hf_YOUR_TOKEN_HERE")


In [36]:
!ls

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


bitsandbytes  fine_tuning_lora_qwen2vl.ipynb  train_ddim.ipynb	       wandb
checkpoints   finetuning_gpt_oss_20b.ipynb    triton_kernel_dev.ipynb


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=ds['train'],
    processing_class=tokenizer,
    )
trainer.train(resume_from_checkpoint=True)

Step,Training Loss
1,0.176600
2,0.206900
3,0.176600
4,0.200400
5,0.197400
6,0.165500
7,0.212500
8,0.198300
9,0.173700
10,0.195100


In [39]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Load base model locally
base_model = AutoModelForCausalLM.from_pretrained(
    "/jupyter-tutorial/hf_models/openai/gpt-oss-20b",  # local path
    local_files_only=True  # tells HF to only look locally
)

# Load LoRA fine-tuned checkpoint
model = PeftModel.from_pretrained(
    base_model,
    "checkpoints/demo/sft",
    local_files_only=True
)

# Merge LoRA weights into base
model = model.merge_and_unload()

# Save merged model locally
model.save_pretrained("gpt-oss-20b-triton-kernel")


MXFP4 quantization requires triton >= 3.4.0 and kernels installed, we will default to dequantizing the model to bf16


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/peft/tuners/lora/layer.py:159: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(


In [40]:
from huggingface_hub import HfApi, HfFolder, Repository

# Set your repo name
repo_id = "Nadiveedishravanreddy/gpt-oss-20b-triton-kernel"

# Save your HF token into colab env if not already
# (Colab usually sets this automatically)
token = HfFolder.get_token()

api = HfApi()
api.create_repo(repo_id, exist_ok=True, token=token)

# Upload model to hub
api.upload_folder(
    folder_path="gpt-oss-20b-triton-kernel",
    repo_id=repo_id,
    repo_type="model",
    token=token
)


model-00002-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00004-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00001-of-00009.safetensors:   0%|          | 0.00/4.50G [00:00<?, ?B/s]

model-00003-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Upload 9 LFS files:   0%|          | 0/9 [00:00<?, ?it/s]

...

model-00006-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00007-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00008-of-00009.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00009-of-00009.safetensors:   0%|          | 0.00/2.75G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Nadiveedishravanreddy/gpt-oss-20b-triton-kernel/commit/8506299e3979c9b4a0c6831d0ccee92836b9c16d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='8506299e3979c9b4a0c6831d0ccee92836b9c16d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nadiveedishravanreddy/gpt-oss-20b-triton-kernel', endpoint='https://huggingface.co', repo_type='model', repo_id='Nadiveedishravanreddy/gpt-oss-20b-triton-kernel'), pr_revision=None, pr_num=None)

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# -----------------------------
# 1️⃣ Path to your merged model
# -----------------------------
local_model_path = "gpt-oss-20b-triton-kernel"  # folder where you saved merged model

# -----------------------------
# 2️⃣ Load tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -----------------------------
# 3️⃣ Load merged model
# -----------------------------
model = AutoModelForCausalLM.from_pretrained(
    local_model_path,
    device_map="auto",       # splits model across available GPUs
    torch_dtype=torch.float16 # FP16 to save memory
)
model.eval()  # set to evaluation mode

# -----------------------------
# 4️⃣ Create text-generation pipeline
# -----------------------------
text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=2048,
    device_map="auto"
)

# -----------------------------
# 5️⃣ Example usage
# -----------------------------
prompt = "Give Triton add kernel"
generated = text_gen_pipeline(
    prompt,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    max_new_tokens=2048
)

print(generated[0]['generated_text'])


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Device set to use cuda:0


Give Triton add kernel" which is a specific example of using Triton's APIs to perform a matrix addition. The code is structured to be run as a script, and it will output the result of adding two matrices.

Let's go through the code step-by-step:

1. The code imports Triton's `Language` and `Scripting` modules, as well as `torch`. It also imports `assert_eq` from `torch.testing._internal.common_utils`. This is likely a function that asserts two tensors are equal.

2. A function `add_kernel` is defined. This function is decorated with `@triton.jit`, which means Triton will compile this function to a GPU kernel. The function signature is `add_kernel(x_ptr, y_ptr, out_ptr, xnumel, XBLOCK : tl.constexpr)`. This means `x_ptr`, `y_ptr`, and `out_ptr` are pointers to the input and output tensors. `xnumel` is the number of elements in the tensors, and `XBLOCK` is a constant expression that will be known at compile time.

3. Inside `add_kernel`, `pid` is calculated as the current program ID (`tl

In [ ]:
prompt = """
Write a Triton kernel for matrix addition. 
The kernel should take two input matrices `A` and `B` and output `C = A + B`.
Use best practices for block size and memory coalescing.
"""

generated = text_gen_pipeline(
    prompt,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    max_new_tokens=2048
)

kernel_code = generated[0]['generated_text']
print(kernel_code)
